# 🚀 ACE-Net Baseline: End-to-End FakeAVCeleb (N=700) Evaluation
### Model: **Adapted ACE-Net with Whisper ASR** (cenet_adapted_whisper.pth)\n### Benchmark: Official 700 FakeAVCeleb clips (350 Real / 350 Fake, Aligned with DeepSentinel)
### Outputs: Saved directly to Google Drive `Baseline_training/checkpoints/`

---

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import torch
from pathlib import Path

drive.mount('/content/drive')

print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name   :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not active! Go to Runtime -> Change runtime type -> T4 GPU.')


## Step 2: Clone or Pull Baseline Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/Baseline_Training'):
    !git clone https://github.com/gjvlio/Baseline_Training.git
%cd /content/Baseline_Training
!git checkout feat/acenet-whisper-eval-700
!git pull
!pip install -q scikit-learn transformers librosa openai-whisper
print('✅ Environment ready!')


## Step 3: Configure Drive Paths & Verify Aligned 700 Manifest
Points directly to `My Drive/Baseline_training` checkpoints and manifests.

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/Baseline_training')
CKPT_PATH = DRIVE_BASE / 'checkpoints' / 'acenet_adapted.pth'
ADAPTED_WHISPER_CKPT = DRIVE_BASE / 'checkpoints' / 'acenet_adapted_whisper.pth'
OUTPUT_CSV = DRIVE_BASE / 'checkpoints' / 'preds_acenet_adapted_700_with_bert.csv'
LOCAL_MANIFEST = Path('Manifests/fakeavceleb_eval_700.csv')
DRIVE_MANIFEST = DRIVE_BASE / 'manifests' / 'fakeavceleb_eval_700.csv'
LOCAL_ADAPT = Path('Manifests/fakeavceleb_adapt_300.csv')
DRIVE_ADAPT = DRIVE_BASE / 'manifests' / 'fakeavceleb_adapt_300.csv'

# Always ensure latest manifests from repo are synced to Drive
if DRIVE_BASE.exists():
    (DRIVE_BASE / 'manifests').mkdir(parents=True, exist_ok=True)
    if LOCAL_MANIFEST.exists():
        shutil.copy(LOCAL_MANIFEST, DRIVE_MANIFEST)
    if LOCAL_ADAPT.exists():
        shutil.copy(LOCAL_ADAPT, DRIVE_ADAPT)
    print(f'Synced manifests to Drive: {DRIVE_BASE / "manifests"}')

print('=' * 75)
print('CONFIGURATION OVERVIEW:')
print(f'  Initial Checkpoint   : {CKPT_PATH} (Exists: {CKPT_PATH.exists()})')
print(f'  Target Eval Manifest : {LOCAL_MANIFEST} (Exists: {LOCAL_MANIFEST.exists()})
')print(f'  Output CSV Path      : {OUTPUT_CSV}')
print('=' * 75)
assert CKPT_PATH.exists(), f'Checkpoint hindi mahanap sa: {CKPT_PATH}'


## Step 4: Fast Selective Unzip of Target Videos (~20-25 Seconds)\nSelectively extracts the clips needed for adaptation (300) and evaluation (700) directly from akeavceleb.zip.

In [ ]:
%cd /content/Baseline_Training

# Awtomatikong i-extract ang 300 adapt clips at 700 eval clips mula sa Drive zip (~25-30s)
!python -m scripts.extract_700_videos \
    --zip-path '/content/drive/MyDrive/Baseline_training' \
    --manifest 'Manifests/fakeavceleb_adapt_300.csv,Manifests/fakeavceleb_eval_700.csv' \
    --output-dir '/content/fakeav_raw'


## Step 4.5: Quick Few-Shot Adaptation with Whisper ASR (5 Epochs, ~2-3 Minutes)
Fine-tunes the classification head on 300 FakeAVCeleb clips using real Whisper transcripts to eliminate train-test distribution shift and prevent zero scores.

In [ ]:
%cd /content/Baseline_Training

# Sanayin ang classification head gamit ang Whisper ASR transcripts (5 Epochs, ~2-3 mins)
!python -u -m scripts.adapt_acenet_fewshot \
    --manifest 'Manifests/fakeavceleb_adapt_300.csv' \
    --raw-dir '/content/fakeav_raw' \
    --init-ckpt '/content/drive/MyDrive/Baseline_training/checkpoints/acenet_adapted.pth' \
    --output-ckpt '/content/drive/MyDrive/Baseline_training/checkpoints/acenet_adapted_whisper.pth' \
    --whisper-model base \
    --epochs 5 \
    --lr 5e-6 \
    --batch-size 16 \
    --device cuda


## Step 5: Run End-to-End Evaluation & Statistical Significance Testing
Runs inference on the exact 700 clips, computes AUC/Accuracy/F1, and performs paired DeLong test vs DeepSentinel.

In [ ]:
%cd /content/Baseline_Training

# Patakbuhin ang evaluator gamit ang bagong Whisper-adapted checkpoint at aligned manifest
!python -u -m scripts.evaluate_fakeavceleb_700 \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --raw-dir '/content/fakeav_raw' \
    --ckpt '/content/drive/MyDrive/Baseline_training/checkpoints/acenet_adapted_whisper.pth' \
    --output-csv '/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_adapted_700_with_bert.csv' \
    --whisper-model base \
    --device cuda


## Step 6: Verify Generated Output Files in Google Drive

In [ ]:
import pandas as pd
from pathlib import Path

preds_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_adapted_700_with_bert.csv')
metrics_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/metrics_preds_acenet_adapted_700_with_bert.csv')

if preds_file.exists():
    df_preds = pd.read_csv(preds_file)
    print(f'✅ Predictions saved successfully! Total rows: {len(df_preds):,}')
    display(df_preds.head(5))

if metrics_file.exists():
    df_metrics = pd.read_csv(metrics_file)
    print('\n📊 OFFICIAL COMPARATIVE METRICS:')
    display(df_metrics)
